# Iterative Retrieval: Accumulate Evidence, Stop Predictably

| Field | Value |
|---|---|
| Stage | Autonomous RAG patterns |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Iterative retrieval needs a support threshold, query history, duplicate guard, and hard attempt budget.

## 30-Second Summary

This notebook starts with `holiday allowance`, records a zero-support attempt, rewrites to `paid leave`, and succeeds on attempt two. An unsupported query exhausts the same two-attempt budget and abstains.

## Why This Matters

Repeating retrieval can bridge terminology, but without history and stopping rules it can issue duplicate searches or loop on an absent answer.

## Scope

| Covers | Does not cover |
|---|---|
| Query history, rewrite, evidence accumulation, duplicate guard, max attempts | Live model rewrite, web fallback, learned support grader |


## Mental Model

```text
retrieve -> score sufficient? yes -> answer
   | no -> new nonduplicate query (attempts left?) -> retrieve | else abstain
```


In [1]:
import re

DOCUMENTS = [
    {"id": "access", "text": "Role access requires manager approval."},
    {"id": "leave", "text": "Employees receive twenty days of paid leave each year."},
]
MAX_ATTEMPTS = 2

def terms(text: str) -> set[str]: return set(re.findall(r"[a-z0-9]+", text.lower())) - {"a", "an", "do", "how", "is", "of", "the", "to"}

def retrieve(query: str) -> tuple[dict, int]:
    return max(((doc, len(terms(query) & terms(doc["text"]))) for doc in DOCUMENTS), key=lambda item: (item[1], item[0]["id"]))


## How It Works

Each attempt appends query, top source, and score. A zero score may trigger a governed rewrite only if the new query is not already in history and the budget remains.


## Baseline

One-shot retrieval for `holiday allowance` has zero support and returns a tie-broken irrelevant document.


In [2]:
baseline_document, baseline_score = retrieve("holiday allowance")
baseline_document, baseline_score


({'id': 'leave',
  'text': 'Employees receive twenty days of paid leave each year.'},
 0)

## Technique Implementation

The loop accumulates trace records and returns a terminal reason. The rewrite map is explicit so the behavior is deterministic and testable.


In [3]:
REWRITES = {"holiday allowance": "paid leave", "weather tomorrow": "weather forecast"}

def iterative_retrieve(initial_query: str) -> dict:
    query, history = initial_query, []
    for attempt in range(1, MAX_ATTEMPTS + 1):
        if query in [item["query"] for item in history]:
            return {"terminal_reason": "duplicate_query", "history": history, "document": None}
        document, score = retrieve(query)
        history.append({"attempt": attempt, "query": query, "document_id": document["id"], "score": score})
        if score > 0:
            return {"terminal_reason": "supported", "history": history, "document": document}
        query = REWRITES.get(query, query + " policy")
    return {"terminal_reason": "budget_exhausted", "history": history, "document": None}

supported = iterative_retrieve("holiday allowance")
supported


{'terminal_reason': 'supported',
 'history': [{'attempt': 1,
   'query': 'holiday allowance',
   'document_id': 'leave',
   'score': 0},
  {'attempt': 2, 'query': 'paid leave', 'document_id': 'leave', 'score': 2}],
 'document': {'id': 'leave',
  'text': 'Employees receive twenty days of paid leave each year.'}}

## Controlled Experiment

We compare a resolvable vocabulary mismatch with an unsupported weather query, checking attempts, query uniqueness, and terminal reasons.


In [4]:
unsupported = iterative_retrieve("weather tomorrow")
results = {"supported": supported, "unsupported": unsupported}
results


{'supported': {'terminal_reason': 'supported',
  'history': [{'attempt': 1,
    'query': 'holiday allowance',
    'document_id': 'leave',
    'score': 0},
   {'attempt': 2, 'query': 'paid leave', 'document_id': 'leave', 'score': 2}],
  'document': {'id': 'leave',
   'text': 'Employees receive twenty days of paid leave each year.'}},
 'unsupported': {'terminal_reason': 'budget_exhausted',
  'history': [{'attempt': 1,
    'query': 'weather tomorrow',
    'document_id': 'leave',
    'score': 0},
   {'attempt': 2,
    'query': 'weather forecast',
    'document_id': 'leave',
    'score': 0}],
  'document': None}}

## Evaluation

The holiday query succeeds on attempt **2** with `leave`. The weather query makes two unique attempts and terminates `budget_exhausted` without an answer. No path can exceed two retrievals.


In [5]:
assert supported["terminal_reason"] == "supported" and supported["document"]["id"] == "leave"
assert len(supported["history"]) == 2 and supported["history"][1]["query"] == "paid leave"
assert unsupported["terminal_reason"] == "budget_exhausted" and len(unsupported["history"]) == MAX_ATTEMPTS
assert all(len({item["query"] for item in run["history"]}) == len(run["history"]) for run in results.values())
print("Iterative retrieval checks passed.")


Iterative retrieval checks passed.


## Decision Guide

| Result | Next action |
|---|---|
| Strong support | Stop and answer |
| Vocabulary mismatch | One bounded rewrite |
| Duplicate query | Stop |
| Budget exhausted | Abstain/fallback with disclosure |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Same query repeats | No history guard | Deduplicate query fingerprints |
| Loop never ends | Missing budget | Max attempts/time/cost |
| Weak doc accepted | Threshold too low | Calibrate support labels |
| Evidence conflicts | Accumulation without reconciliation | Provenance-aware synthesis |


## Production Notes

### Observability
Log attempt, query hash/text per policy, ranked IDs/scores, accumulated evidence, and terminal reason.

### Safety and Guardrails
Rewrites cannot broaden authorization or silently add web sources.

### Latency and Cost
Budget worst-case attempts and stop early on sufficient support.


## Practice

Add a rewrite that returns its original query and verify the duplicate guard stops before another retrieval.

## Recall

Toggle - Recall: What prevents infinite retrieval?
Attempt budget plus duplicate-query detection.

Toggle - Recall: When should the loop stop early?
As soon as evidence meets the support threshold.

## Sources

- [LangGraph loops and branches](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- Repository-owned synthetic policy data

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for bounded deterministic iteration | Calibrate support thresholds on labeled failures |
